In [ ]:
import easydict
import torch
import yaml
from chromadb import PersistentClient
from tqdm import tqdm

from models.Point_MAE import Point_MAE
from tools import builder

In [ ]:
config_path='cfgs/pretrain.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
config = easydict.EasyDict(config['model'])  # Directly convert model config section to EasyDict

if config is None:
    raise ValueError("Config is None")

In [ ]:
model_path="data/weights/pretrain.pth"
model = Point_MAE(config=config)  # Config will be loaded from checkpoint
builder.load_model(model, ckpt_path=model_path)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

In [ ]:
from datasets.DrivaernetDataset import get_dataloaders
data_path = "data/updated_point_clouds.db"
batch_size = 16
num_workers = 4
train_dataloader, _, _ = get_dataloaders(
    db_path=data_path,
    batch_size=batch_size,
    num_workers=num_workers
)

In [ ]:
first_batch = next(iter(train_dataloader))
points = first_batch['points']

In [ ]:
# Encode points
# encoded = model.encode_pts(points[:, :, :3].contiguous()) # this takes in B N 3

# # Convert to numpy and add to ChromaDB
# embeddings = encoded.cpu().numpy() # B C